# **Model**

# **synthetic spectra database**

In [43]:
# generating synthetic spectra with different noise levels

from synthetic import generate_synthetic_spectral_data

config = [
    {
        'nome': 'A',
        'n_amostras': 156,
        'picos': [250, 550, 700, 850],  # 4 picos
        'amp_media': 1.0,
        'amp_std': 0.3,
        'larg_media': 15.0,
        'larg_std': 2.0,
        'ruido_std': 0.04
    },
    {
        'nome': 'B',
        'n_amostras': 146,
        'picos': [50, 250, 700, 850],  # 3 picos (sem pico em 550)
        'amp_media': 1.3,
        'amp_std': 0.3,
        'larg_media': 15.0,
        'larg_std': 1.8,
        'ruido_std': 0.035
    }
]

data_complete = generate_synthetic_spectral_data(
    configuracao_classes=config,
    n_pontos=500,
    x_min=1,
    x_max=1000,
    seed=0
)

import pandas as pd
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 
data_complete.iloc[:, 1:].T.plot()

In [44]:
# importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks


data = data_complete.iloc[:, 1:]
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.iloc[:, 1:], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.iloc[:, 1:], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass  = prepr.mc(Xcalclass)
Xpredclass_prep = Xpredclass - mean_calclass

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=1,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-12 10:42:59,259 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

2026-01-12 10:42:59,286 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,1.0,1.0,1.0,"[[109, 0], [0, 102]]",1.0,1.0,1.0,"[[109, 0], [0, 102]]",1.0,1.0,1.0,"[[47, 0], [0, 44]]",64.357865,47.337063,64.357865,47.337063


# **VIP and SHAP**

In [45]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('F1', 1.0, 100.0),
('background1', 100.0, 200.0),
('F2', 200.0, 300.0),
('background2', 300.0, 500.0),
('F3', 500.0, 600.0),
('background3', 600.0, 660.0),
('F4', 660.0, 750.0),
('background4', 750.0, 815.0),
('F5', 815.0, 890.0),
('background4', 890.0, 1000.0)
]

In [46]:
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 
plsda_results[4].T.plot()

In [47]:
# calculando a covariancia entre cada variável espectral e a predição do modelo PLS-DA
cov_scores = []
y_pred = plsda_results[5].iloc[:,-1].values # using the continuous predictions from LV=3
for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [48]:
import numpy as np
import pandas as pd

# vip
vip_scores_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'VIP_Score' : plsda_results[4].T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_vip = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in vip_scores_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_vip[i] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# reg vet
reg_vet = pd.DataFrame(plsda_results[3].coef_, columns=plsda_results[3].feature_names_in_) # creating a DataFrame with regression coefficients
reg_vet = reg_vet.T
reg_vet.insert(0, 'energy', reg_vet.index) # adding energy column
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy', 'Reg_coef'] # renaming
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs() # adding absolute value column
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True) # sorting by absolute value

# gerando uma nova coluna em reg_vet com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_reg = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
    for i in reg_vet['energy']: # iterando sobre cada valor de energia no reg_vet
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_reg[i] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_reg
reg_vet

# vamos filtrar reg_vet para manter apenas as zonas espectrais únicas com maior valor absoluto do coeficiente de regressão
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [ ]:
# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# # 15 MIN    

Using 211 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
100%|██████████| 211/211 [21:16<00:00,  6.05s/it]


In [32]:
vip_scores_unique_df

,energy,VIP_Score,Zone
0,49.048096192384776,4.592658,F1
1,549.5490981963928,3.540725,F3
2,249.248496993988,1.223649,F2
3,699.6993987975952,1.102782,F4
4,849.8496993987976,1.048931,F5
5,813.813627254509,0.074429,background4
6,601.6012024048097,0.030630,background3
7,101.10020040080161,0.029941,background1
8,499.498997995992,0.028933,background2


In [21]:
# vip_scores_unique_df.to_csv('Synthetic_databases/plsda/vip_scores_soil.csv', index=False, sep=';')
# reg_vet_unique_df.to_csv('Synthetic_databases/plsda/reg_vet_soil.csv', index=False, sep=';')
# shap_unique_df.to_csv('Synthetic_databases/plsda/shap_soil.csv', index=False, sep=';')

In [49]:
shap_unique_df = pd.read_csv('Synthetic_databases/plsda/shap_soil.csv', sep=';') # loading previously saved shap_unique_df
#vip_scores_unique_df = pd.read_csv('XRF_databases/soil/plsda/vip_scores_soil.csv', sep=';') # loading previously saved vip_scores_unique_df#
#reg_vet_unique_df = pd.read_csv('XRF_databases/soil/plsda/reg_vet_soil.csv', sep=';') # loading previously saved reg_vet_unique_df

# **SMeX**

In [50]:
def aggregate_spectral_zonesv2(spectral_zones_dict, aggregator='sum'):
    """
    Agrega os valores das zonas espectrais usando diferentes funções de agregação.
    
    Esta função processa cada zona espectral (DataFrame com múltiplas colunas de energia)
    e reduz cada linha (amostra) a um único valor numérico usando a função de agregação
    especificada.
    
    Parameters
    ----------
    - **spectral_zones_dict** : dict
        Dicionário retornado por extract_spectral_zones, onde:
        - chaves = nomes das zonas espectrais (ex: 'Ca ka', 'Fe ka')
        - valores = DataFrames com dados espectrais (linhas=amostras, colunas=energias)
    
    - **aggregator** : str, opcional (padrão='sum')
        Função de agregação a aplicar nas colunas de cada zona. Opções:
        - **'sum'**: Soma de todos os valores da zona (padrão)
        - **'mean'**: Média aritmética dos valores
        - **'median'**: Mediana dos valores
        - **'max'**: Valor máximo na zona
        - **'min'**: Valor mínimo na zona
        - **'std'**: Desvio padrão dos valores
        - **'var'**: Variância dos valores
        - **'extreme'**: Valor de maior magnitude (mais intenso) na zona, ou seja,
          escolhe o valor com maior valor absoluto em cada amostra (pode ser positivo ou negativo)
    
    Returns
    -------
    - **aggregated_df** : pd.DataFrame
        DataFrame com valores agregados, onde:
        - linhas = amostras (mesmo índice dos DataFrames originais)
        - colunas = zonas espectrais
        - valores = resultado da agregação (mesmo formato que .sum(axis=1))
    
    Raises
    ------
    - ValueError
        Se o agregador especificado não for reconhecido.
    """
    import pandas as pd
    import numpy as np
    
    # VALIDAÇÃO DE ENTRADA
    valid_aggregators = ['sum', 'mean', 'median', 'max', 'min', 'std', 'var', 'extreme']
    
    if aggregator not in valid_aggregators:
        raise ValueError(
            f"Agregador '{aggregator}' não reconhecido.\n"
            f"Opções válidas: {', '.join(valid_aggregators)}"
        )
    
    # MAPEAMENTO DOS AGREGADORES
    # Dicionário que mapeia strings para funções do pandas
    aggregation_functions = {
        'sum': lambda df: df.sum(axis=1),        # soma ao longo das colunas
        'mean': lambda df: df.mean(axis=1),      # média
        'median': lambda df: df.median(axis=1),  # mediana
        'max': lambda df: df.max(axis=1),        # valor máximo
        'min': lambda df: df.min(axis=1),        # valor mínimo
        'std': lambda df: df.std(axis=1),        # desvio padrão
        'var': lambda df: df.var(axis=1),        # variância
        # 'extreme': escolhe o valor com maior magnitude (abs), preservando o sinal
        'extreme': lambda df: df.apply(
            lambda row: (row.loc[row.abs().idxmax()] if row.notna().any() else np.nan),
            axis=1
        ),
    }
    
    # AGREGAÇÃO DAS ZONAS ESPECTRAIS
    aggregated_dict = {}  # dicionário para armazenar resultados
    
    for zone_name, zone_df in spectral_zones_dict.items():
        # Aplica a função de agregação selecionada
        # O resultado é uma Series (mesma estrutura que .sum(axis=1))
        aggregated_series = aggregation_functions[aggregator](zone_df)
        
        # Armazena no dicionário
        aggregated_dict[zone_name] = aggregated_series
    
    # CONSTRUÇÃO DO DATAFRAME FINAL
    # Cada chave vira uma coluna, preservando os índices originais
    aggregated_df = pd.DataFrame(aggregated_dict)    
    return aggregated_df

In [53]:
from explaining import extract_spectral_zones
spectral_zones_class = extract_spectral_zones(Xcalclass_prep, spectral_cuts) # extracting the spectral zones
zone_sums_df = aggregate_spectral_zonesv2(spectral_zones_class, aggregator='extreme')

# vamos calcular agora a covariancia entre as zonas espectrais e a predição do modelo PLS-R
# usando as zonas agregadas já calculadas em 'zone_sums_df'
zone_cov_scores = []
for zone in zone_sums_df.columns:
    cov_score = np.abs(np.cov(zone_sums_df[zone].values, plsda_results[5].iloc[:,-1].values))[0, 1]  # covariância entre a zona espectral agregada e as predições do PLS-R
    zone_cov_scores.append(cov_score)
zone_cov_scores_df = pd.DataFrame({
    'Zone': zone_sums_df.columns,
    'Cov_Score': zone_cov_scores
})

zone_cov_scores_df = zone_cov_scores_df.sort_values(by='Cov_Score', ascending=False).reset_index(drop=True)
zone_cov_scores_df

,Zone,Cov_Score
0,F1,0.336952
1,F3,0.263968
2,F2,0.098491
3,F4,0.085727
4,F5,0.085344
5,background3,0.008155
6,background1,0.003340
7,background2,0.001451
8,background4,0.000888


In [54]:
from explaining import extract_spectral_zones
from explaining import aggregate_spectral_zones
from explaining import predicates_by_quantiles
from explaining import create_predicate_info_dict
from explaining import bagging_predicates, calculate_predicate_metrics
from explaining import build_predicate_graph
import numpy as np
import pandas as pd

spectral_zones_class = extract_spectral_zones(Xcalclass_prep, spectral_cuts) # extracting the spectral zones
#zone_sums_df = aggregate_spectral_zones(spectral_zones_class, aggregator='sum')
zone_sums_df = aggregate_spectral_zonesv2(spectral_zones_class, aggregator='extreme')
predicates_quantiles = predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8]) # getting predicates for quartiles
co_occurrence_matrix_df=predicates_quantiles[2]

# Criar dicionário de informações
predicate_info_dict = create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=plsda_results[5].iloc[:, -1]
)

# LISTA DE SEMENTES A TESTAR

random_seeds = [0, 1, 42]

all_results = {}

training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    
    # Bagging
    bags_result_seed = bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=60,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    mi_results_dict_seed = calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)

# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente
    DG = build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# vamos calcular a LRC de acordo com as diferentes sementes
import networkx as nx

lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    local_reaching_centrality = {
        node: nx.local_reaching_centrality(DG, node, weight='weight') 
        for node in DG.nodes()
    }

    # Ordenar por LRC
    sorted_lrc = sorted(local_reaching_centrality.items(), key=lambda x: x[1], reverse=True)
    
    # Criar DataFrame com LRC
    lrc_df_seed = pd.DataFrame(sorted_lrc, columns=['Node', 'Local_Reaching_Centrality'])
    
    # Extrair informações dos predicados (zona, threshold, operador)
    zones = []
    thresholds = []
    operators = []
    
    for node in lrc_df_seed['Node']:
        if node.startswith('Class_'):
            zones.append(None)
            thresholds.append(None)
            operators.append(None)
        else:
            pred_row = predicates_quantiles[0][predicates_quantiles[0]['rule'] == node].iloc[0]
            zones.append(pred_row['zone'])
            thresholds.append(pred_row['thresholds'])
            operators.append(pred_row['operator'])
    
    lrc_df_seed['Zone'] = zones
    lrc_df_seed['Threshold'] = thresholds
    lrc_df_seed['Operator'] = operators
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    
    # Armazenar LRC
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe

lrc_all_seeds_df = pd.DataFrame() # 
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed]
    lrc_df_seed = lrc_df_seed.rename(columns={
        'Node': f'Predicate_Seed_{seed}'
    })
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df.head(20) # exibindo o dataframe consolidado com predicados de todas as sementes   


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 54 | Descartados: 18
Bag_13 | Amo

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,F1 > -0.67,F1 > -0.67,F1 > -0.67
1,F3 > -0.56,F3 > -0.56,F3 > -0.56
2,F1 <= 0.80,F1 > -0.64,F1 > -0.64
3,F1 > -0.64,F1 <= 0.80,F1 <= 0.80
4,F3 <= 0.57,F3 <= 0.57,F3 <= 0.57
5,F3 <= 0.30,F3 > -0.53,F3 <= 0.30
6,F3 > -0.53,F3 <= 0.30,F2 > -0.35
7,F2 > -0.35,F2 > -0.35,F3 > -0.53
8,F1 <= 0.41,F1 <= 0.41,F5 > -0.34
9,F2 <= 0.35,F5 > -0.34,F1 <= 0.41


In [28]:
# vamos conferir se existe alguma aresta com peso igual a zero 

zero_weight_edges_by_seed = {}

for seed in random_seeds:
    DG = graphs_by_seed[seed]
    zero_weight_edges = []
    
    for u, v, data in DG.edges(data=True):
        weight = data.get('weight', 0)
        if weight == 0:
            zero_weight_edges.append((u, v, weight))
    
    zero_weight_edges_by_seed[seed] = zero_weight_edges
    
    print(f"Semente {seed}:")
    print(f"  Total de arestas: {DG.number_of_edges()}")
    print(f"  Arestas com peso zero: {len(zero_weight_edges)}")
    
    if zero_weight_edges:
        print(f"  Exemplos de arestas com peso zero:")
        for u, v, w in zero_weight_edges[:5]:  # Mostrar até 5 exemplos
            print(f"    {u} -> {v} (peso: {w})")
    print()

# Resumo geral
total_zero_edges = sum(len(edges) for edges in zero_weight_edges_by_seed.values())
print(f"\nResumo Geral:")
print(f"  Total de arestas com peso zero em todas as sementes: {total_zero_edges}")

if total_zero_edges == 0:
    print("  ✓ Não existem arestas com peso zero em nenhum dos grafos!")
else:
    print("  ⚠ Existem arestas com peso zero que podem causar problemas em cálculos de centralidade.")

Semente 0:
  Total de arestas: 468
  Arestas com peso zero: 17
  Exemplos de arestas com peso zero:
    F3 <= 0.29 -> F3 > 0.60 (peso: 0.0)
    F1 <= 0.44 -> F1 > 0.44 (peso: 0.0)
    F1 > 0.44 -> F3 > 0.29 (peso: 0.0)
    F3 > 0.29 -> F3 <= 0.29 (peso: 0.0)
    F3 > 0.29 -> F1 > 0.73 (peso: 0.0)

Semente 1:
  Total de arestas: 471
  Arestas com peso zero: 17
  Exemplos de arestas com peso zero:
    F4 > 0.12 -> F4 <= 0.12 (peso: 0.0)
    F5 > 0.14 -> F5 <= 0.14 (peso: 0.0)
    F1 > 0.44 -> F1 <= 0.44 (peso: 0.0)
    F1 > 0.44 -> F3 > 0.60 (peso: 0.0)
    F2 > 0.05 -> F2 <= 0.05 (peso: 0.0)

Semente 42:
  Total de arestas: 456
  Arestas com peso zero: 17
  Exemplos de arestas com peso zero:
    F1 <= 0.44 -> F1 > 0.73 (peso: 0.0)
    F3 <= 0.29 -> F3 > 0.29 (peso: 0.0)
    F3 > 0.29 -> F1 > 0.44 (peso: 0.0)
    F1 > 0.44 -> F3 > 0.60 (peso: 0.0)
    F1 > 0.44 -> F1 <= 0.44 (peso: 0.0)


Resumo Geral:
  Total de arestas com peso zero em todas as sementes: 51
  ⚠ Existem arestas com peso

In [55]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:9].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:9].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:9].values
features_importance.head(9)

,Vip,Reg_coef,Shap,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,F1,F1,F1,F1,F1,F1
1,F3,F3,F3,F3,F3,F3
2,F2,F2,F4,F2,F2,F2
3,F4,F4,background3,F5,F5,F5
4,F5,F5,background4,F4,F4,F4
5,background4,background4,F5,background4,background4,background4
6,background2,background2,background1,background2,background1,background2
7,background3,background3,F2,background1,background2,background1
8,background1,background1,background2,background3,background3,background3


## **RBO**

In [56]:
# utilizando o Rank-Biased Overlap (RBO) para comparar as listas de importância de características tendo o vip como referencia
# com p = 1 é rbo equivalente ao overlap simples (interseção sobre união) sem peso para posições iniciais
# quanto menor o p, mais peso é dado para as posições iniciais da lista (mais relevante para nosso caso)
import rbo

rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_Seed_{seed}' for seed in random_seeds] #'Shap'] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10) # o p=0.9 dá mais peso para as posições iniciais, k=10 limita a comparação às top 10 posições
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method', 'RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
0,Vip,Reg_coef,0.959646
2,Vip,LRC_Seed_0,0.930833
4,Vip,LRC_Seed_42,0.930833
3,Vip,LRC_Seed_1,0.925791
1,Vip,Shap,0.834533


In [57]:
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        list_1 = features_importance[method_1].tolist()
        list_2 = features_importance[method_2].tolist()
        rbo_score = rbo.RankingSimilarity(list_1, list_2).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_4136\43539824.py:10: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
0,Vip,Reg_coef,0.959646
13,LRC_Seed_0,LRC_Seed_42,0.959646
12,LRC_Seed_0,LRC_Seed_1,0.954604
14,LRC_Seed_1,LRC_Seed_42,0.954604
2,Vip,LRC_Seed_0,0.930833
4,Vip,LRC_Seed_42,0.930833
6,Reg_coef,LRC_Seed_0,0.930833
8,Reg_coef,LRC_Seed_42,0.930833
3,Vip,LRC_Seed_1,0.925791
7,Reg_coef,LRC_Seed_1,0.925791
